In [1]:
# Imports
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import LongformerTokenizer, LongformerForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
import xgboost as xgb
import warnings
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score 
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
import os
from torch.optim import AdamW

# Ignore specific FutureWarning
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost.data')

### Finetune

In [7]:
base = pd.read_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_HH.csv')

In [8]:
#nurse_notes = pd.read_csv(f'/datafs_a/haim14_data/blood_infection/notes_matched_nurse_7.csv')

In [ ]:
notes = pd.read_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/notes_matched.csv')

In [10]:
unique_patients = base[['PatientDurableKey']].drop_duplicates()

In [15]:
# MAKE THE TRAIN, TEST, VAL SPLITS
# 80 train, 20 test
SEED = 14
from sklearn.model_selection import train_test_split
train,val=train_test_split(unique_patients, test_size=0.2, train_size=0.8, random_state=None, shuffle=True)
train = train.merge(base, how = "left", on=['PatientDurableKey'])
val = val.merge(base, how = "left", on=['PatientDurableKey'])
#train.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_train.csv")
#test.to_csv("/datafs_a/carolgao/haim14_data/blood_infection/base_test.csv")

In [2]:
# Define the model for fine-tuning
class FineTunedModel(nn.Module):
    def __init__(self, pretrained_model_name, num_labels):
        super(FineTunedModel, self).__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name, num_labels=num_labels)

        # Optionally freeze all layers but the last N
        n_layers_to_freeze = self.bert.config.num_hidden_layers - 1  # Freezing all but the last 1 layers

        # Freezing layers
        for name, param in self.bert.named_parameters():
            # Freeze parameters except the last N layers
            if 'encoder.layer' in name:
                layer_index = int(name.split('.')[3])
                if layer_index < n_layers_to_freeze:
                    param.requires_grad = False
                
    def forward(self, input_ids, attention_mask, labels=None, return_embedding=False):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, labels=labels, output_hidden_states=True)
        if return_embedding:
            # Return the embeddings from the last hidden state
            # Typically, the first token ([CLS] token) is used as the aggregate sequence representation.
            embeddings = outputs.hidden_states[-1][:, 0, :]  # Get the last layer's [CLS] token state
            return embeddings
        return outputs
    
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, return_tensors='pt', padding='max_length', truncation=True, max_length=self.max_length)
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [16]:
# H&P notes
df_notes_train = train[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CombinedNotes']].drop_duplicates()
df_notes_val = val[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CombinedNotes']].drop_duplicates()
df_notes_train['CombinedNotes'].mask(df_notes_train['CombinedNotes'].isnull() == True, "", inplace=True)
df_notes_val['CombinedNotes'].mask(df_notes_val['CombinedNotes'].isnull() == True, "", inplace=True)
assert len(df_notes_train) == len(train)
assert len(df_notes_val) == len(val)

In [5]:
# Nurse notes 
df_nurse_notes_train = train[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'NurseNotes']].drop_duplicates()
df_nurse_notes_val = val[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'NurseNotes']].drop_duplicates()
df_nurse_notes_train['NurseNotes'].mask(df_nurse_notes_train['NurseNotes'].isnull() == True, "", inplace=True)
df_nurse_notes_val['NurseNotes'].mask(df_nurse_notes_val['NurseNotes'].isnull() == True, "", inplace=True)
assert len(df_nurse_notes_train) == len(train)
assert len(df_nurse_notes_val) == len(val)

In [19]:
y_notes_train = train['Positive'].tolist()
y_notes_val = val['Positive'].tolist()

In [3]:
def evaluate_with_auc(model, test_loader, device):
    from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
    import torch

    model.eval()  # Set model to evaluation mode
    predictions = []
    true_labels = []

    with torch.no_grad():  # Disable gradient calculation for evaluation
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass to get logits
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            # Get predicted probabilities for the positive class
            probs = torch.softmax(logits, dim=1)[:, 1]  # Probabilities for class 1 (positive class)
            predictions.extend(probs.cpu().numpy())  # Convert to numpy and extend the list
            true_labels.extend(labels.cpu().numpy())  # Convert to numpy and extend the list

    # Convert lists to numpy arrays
    true_labels = np.array(true_labels)
    predictions = np.array(predictions)

    # Calculate AUC
    auc = roc_auc_score(true_labels, predictions)

    # Convert predicted probabilities to binary predictions using 0.5 threshold
    binary_predictions = (predictions >= 0.5).astype(int)

    # Calculate other metrics
    accuracy = accuracy_score(true_labels, binary_predictions)
    precision = precision_score(true_labels, binary_predictions)
    recall = recall_score(true_labels, binary_predictions)
    f1 = f1_score(true_labels, binary_predictions)

    return accuracy, precision, recall, f1, auc

In [4]:
def fine_tune_fit_and_save_model(df_notes_train, df_notes_test, y_notes_train, y_notes_test, notes_type, gpu_name="cuda:0"):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    from torch.optim import AdamW
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from sklearn.metrics import roc_auc_score
    import os
    from torch.cuda.amp import GradScaler, autocast

    tokenizer = AutoTokenizer.from_pretrained('/datafs_a/carolgao/Clinical-Longformer')

    # Prepare DataLoaders
    batch_size = 1  # Keep batch size small to save memory
    train_dataset = TextDataset(df_notes_train[notes_type].tolist(), y_notes_train, tokenizer)
    test_dataset = TextDataset(df_notes_test[notes_type].tolist(), y_notes_test, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    # Initialize the model
    model = FineTunedModel("/datafs_a/carolgao/Clinical-Longformer", num_labels=2)
    
    # Move model to GPU if available
    device = torch.device(gpu_name if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Training setup
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
    loss_fn = nn.CrossEntropyLoss()
    
    # Initialize GradScaler for mixed precision training
    scaler = GradScaler()

    # Training loop
    epochs = 1
    accumulation_steps = 4  # Adjust based on available memory

    # Directory to save models
    save_dir = '/datafs_a/carolgao/haim14_data/models/blood_infection/'
    os.makedirs(save_dir, exist_ok=True)

    # Training loop with gradient accumulation, mixed precision, cache clearing, and model saving
    for epoch in range(epochs):
        model.train()  # Set model to training mode
        total_loss = 0
        optimizer.zero_grad()  # Initialize gradients

        for i, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass with mixed precision
            with autocast():
                outputs = model(input_ids, attention_mask, labels=labels)
                loss = outputs.loss / accumulation_steps  # Normalize loss

            # Backward pass with mixed precision
            scaler.scale(loss).backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            # Clear GPU memory
            total_loss += loss.item() * accumulation_steps
            del input_ids, attention_mask, labels, outputs
            torch.cuda.empty_cache()

        avg_train_loss = total_loss / len(train_loader)

        # Validation
        accuracy, precision, recall, f1, auc = evaluate_with_auc(model, test_loader, device)
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Testing Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")

        # Save the model after each epoch using AUC as part of the filename
        model_save_path = os.path.join(save_dir, f"{notes_type}_model_epoch_{epoch + 1}.pt")
        torch.save(model.state_dict(), model_save_path)
        print(f"Model saved to {model_save_path}")

In [28]:
# H&P Notes
should_break = False  # Initialize the flag before the loop
max_retries = 10  # Optional: Limit retries to prevent infinite looping

while not should_break and max_retries > 0:
    for i in range(3,4):
        gpu_name = f"cuda:{i}"
        try:
            fine_tune_fit_and_save_model(df_notes_train, df_notes_val, y_notes_train, y_notes_val, "CombinedNotes", gpu_name)
            should_break = True  # Set flag to break both loops
            break  # Breaks out of the for loop
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print(f"GPU {i} out of memory, trying {i+1}")
                torch.cuda.empty_cache()  # Free GPU memory before moving to the next GPU
            else:
                raise e  # Raise the error if it's not related to CUDA memory
    
    max_retries -= 1  # Decrement the retry counter

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/carolgao/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Initializing global attention on CLS token...


Epoch 1/1 | Train Loss: 0.2961 | Testing Accuracy: 0.9122, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.6511
Model saved to /datafs_a/carolgao/haim14_data/models/blood_infection/CombinedNotes_model_epoch_1.pt


In [ ]:
# Nurse notes 
should_break = False  # Initialize the flag before the loop
max_retries = 20  # Optional: Limit retries to prevent infinite looping

while not should_break and max_retries > 0:
    for i in range(3,4):
        gpu_name = f"cuda:{i}"
        try:
            fine_tune_fit_and_save_model(df_nurse_notes_train, df_nurse_notes_val, y_notes_train, y_notes_val, "NurseNotes", gpu_name)
            should_break = True  # Set flag to break both loops
            break  # Breaks out of the for loop
        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print(f"GPU {i} out of memory, trying {i+1}")
                torch.cuda.empty_cache()  # Free GPU memory before moving to the next GPU
            else:
                raise e  # Raise the error if it's not related to CUDA memory
    
    max_retries -= 1  # Decrement the retry counter

/datafs_a/carolgao/anaconda3/envs/new_env/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/LLMs/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Initializing global attention on CLS token...


GPU 3 out of memory, trying 4


Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/LLMs/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPU 3 out of memory, trying 4


Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/LLMs/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/LLMs/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPU 3 out of memory, trying 4


Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/LLMs/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GPU 3 out of memory, trying 4


Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at /datafs_a/LLMs/Clinical-Longformer and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Generate embeddings 

In [5]:
df_notes = pd.DataFrame()
for k in ['HH', 'BH', 'HOCC', 'MMC']:
    df_notes_test_k = pd.read_csv(f"/datafs_a/carolgao/haim14_data/blood_infection/base_{k}_raw_notes.csv")
    #df_notes_test_k = test_k[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay', 'CombinedNotes']].drop_duplicates()
    df_notes_test_k['CombinedNotes'].mask(df_notes_test_k['CombinedNotes'].isnull() == True, "", inplace=True)
    df_notes = pd.concat([df_notes, df_notes_test_k])

df_notes = df_notes.reset_index(drop = True)

In [10]:
# Enable CUDA launch blocking for debugging (optional)
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Step 1: Initialize the model architecture
model_path = "/datafs_a/carolgao/Clinical-Longformer"  # The base model used for fine-tuning
tokenizer = LongformerTokenizer.from_pretrained(model_path)
#model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)
model = FineTunedModel(model_path, num_labels=2)


should_break = False  
# Initialize the flag before the loop
max_retries = 10  # Optional: Limit retries to prevent infinite looping
while not should_break and max_retries > 0:
    for i in range(1,2):
        gpu_name = f"cuda:{i}"
        try:
            # Step 3: Move the model to the appropriate device (GPU or CPU)
            device = torch.device(gpu_name if torch.cuda.is_available() else "cpu")
            model.to(device)  
            
            # Now the model is ready for inference or further fine-tuning
            model.eval()  # Set the model to evaluation mode if using it for inference

            print("Model loaded successfully and ready for inference.")
            should_break = True  # Set flag to break both loops
            break  # Breaks out of the for loop
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"GPU {i} out of memory, trying {i+1}")
                torch.cuda.empty_cache()  # Free GPU memory before moving to the next GPU
            else:
                raise e  # Raise the error if it's not related to CUDA memory
    
    max_retries -= 1  # Decrement the retry counter

# Step 2: Load the saved state dictionary
checkpoint_path = f"/datafs_a/carolgao/haim14_data/models/blood_infection/CombinedNotes_model_epoch_1.pt"  # Replace with your .pt file path
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)

# Assume 'model' is your fine-tuned Clinical-Longformer
model.eval()  # Set the model to evaluation mode

# Prepare the data
#text_list = df_notes["CombinedNotes"].astype(str).tolist()
#labels = df_notes["Positive_bin"].tolist()  # Convert classes to binary labels

#dataset = TextDataset(text_list, labels, tokenizer)
#batch_size = 1  # Use batch size 1 to avoid memory issues

#test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)


Model loaded successfully and ready for inference.


FineTunedModel(
  (bert): LongformerForSequenceClassification(
    (longformer): LongformerModel(
      (embeddings): LongformerEmbeddings(
        (word_embeddings): Embedding(50265, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (position_embeddings): Embedding(4098, 768, padding_idx=1)
      )
      (encoder): LongformerEncoder(
        (layer): ModuleList(
          (0-11): 12 x LongformerLayer(
            (attention): LongformerAttention(
              (self): LongformerSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (query_global): Linear(in_features=768, out_features=768, bias=True)
                (key_global): L

In [11]:
def evaluate_with_auc(model, test_loader, device):
    from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
    import torch

    model.eval()  # Set model to evaluation mode
    predicted_probs = []
    true_labels = []
    last_hidden_states = []

    with torch.no_grad():  # Disable gradient calculation for evaluation
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass to get logits
            #outputs = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            logits = outputs.logits

            # Extract the hidden states
            hidden_states = outputs.hidden_states  # A tuple containing all hidden layers
            last_hidden_state = hidden_states[-1]  # Extract the last hidden layer
        
            # Get predicted probabilities for the positive class
            probs = torch.softmax(logits, dim=1)[:, 1]  # Probabilities for class 1 (positive class)
            predicted_probs.extend(probs.cpu().numpy())  # Convert to numpy and extend the list
            true_labels.extend(labels.cpu().numpy())  # Convert to numpy and extend the list
            last_hidden_states.append(last_hidden_state.cpu().numpy()) # Store last hidden state

    # Convert lists to numpy arrays
    true_labels = np.array(true_labels)
    predicted_probs = np.array(predicted_probs)

    # Calculate AUC
    auc = roc_auc_score(true_labels, predicted_probs)

    # Convert predicted probabilities to binary predictions using 0.5 threshold
    binary_predictions = (predicted_probs >= 0.5).astype(int)
    # Calculate other metrics
    accuracy = accuracy_score(true_labels, binary_predictions)
    precision = precision_score(true_labels, binary_predictions)
    recall = recall_score(true_labels, binary_predictions)
    f1 = f1_score(true_labels, binary_predictions)
    
    print(f"Testing Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
    
    return auc, predicted_probs, last_hidden_states

In [12]:
round(len(df_notes)/100) +1 

975

In [13]:
final_predicted_probs = np.zeros([0])
embeddings = np.zeros([0, 768])
for k in range(round(len(df_notes)/1000) +1 ):
    df_notes_temp = df_notes.iloc[1000*k:min(1000*(k+1), len(df_notes))]
    # Prepare the data
    text_list = df_notes_temp["CombinedNotes"].astype(str).tolist()
    labels = df_notes_temp["Positive"].tolist()  # Convert classes to binary labels

    dataset = TextDataset(text_list, labels, tokenizer)
    batch_size = 1  # Use batch size 1 to avoid memory issues
    
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    # Set the transformers logger level to WARNING or ERROR to suppress INFO messages
    import logging
    logging.getLogger("transformers").setLevel(logging.ERROR)
    
    auc, predicted_probs, last_hidden_states = evaluate_with_auc(model, test_loader, device)
    
    embeddings_flattened = np.zeros([len(df_notes_temp), 768])
    for i in range(len(df_notes_temp)):
        embeddings_flattened[i,:] = np.mean(last_hidden_states[i][0,:], axis=0)
    
    final_predicted_probs = np.concatenate((final_predicted_probs, predicted_probs))
    embeddings = np.concatenate((embeddings, embeddings_flattened))

    print("finished", k)

    pd.DataFrame(final_predicted_probs).to_csv("/datafs_a/carolgao/haim14_data/blood_infection/hp_notes_predicted_probs.csv", index = False)
    pd.DataFrame(embeddings).to_csv("/datafs_a/carolgao/haim14_data/blood_infection/hp_notes_embeddings.csv", index = False)

/datafs_a/carolgao/anaconda3/envs/Torch_8P/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Testing Accuracy: 0.9080, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.7134
finished 0
Testing Accuracy: 0.9130, Precision: 1.0000, Recall: 0.0114, F1: 0.0225, AUC: 0.6281
finished 1


/datafs_a/carolgao/anaconda3/envs/Torch_8P/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Testing Accuracy: 0.9200, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.6943
finished 2
Testing Accuracy: 0.9300, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.6317
finished 3


/datafs_a/carolgao/anaconda3/envs/Torch_8P/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


...

Testing Accuracy: 0.8960, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.5744
finished 95


/datafs_a/carolgao/anaconda3/envs/Torch_8P/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Testing Accuracy: 0.9030, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.5828
finished 96


/datafs_a/carolgao/anaconda3/envs/Torch_8P/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Testing Accuracy: 0.8946, Precision: 0.0000, Recall: 0.0000, F1: 0.0000, AUC: 0.5978
finished 97


In [14]:
notes_embeddings = pd.read_csv('/datafs_a/carolgao/haim14_data/blood_infection/hp_notes_embeddings.csv')
notes_embeddings.columns = ['emb_' + col for col in notes_embeddings.columns]

In [25]:
probs = pd.read_csv('/datafs_a/carolgao/haim14_data/blood_infection/hp_notes_predicted_probs.csv')
probs.columns = ['probs']

In [30]:
df_notes = df_notes.reset_index()
df_with_embeddings = pd.concat([df_notes, notes_embeddings], axis=1)
df_with_embeddings = df_with_embeddings.drop(columns = ['index', 'Unnamed: 0.1', 'Unnamed: 0'])

In [31]:
df_with_embeddings = pd.concat([df_with_embeddings, probs],axis=1)

In [33]:
for k in ["HH", "BH", "HOCC", "MMC"]:
    df_temp = df_with_embeddings.loc[df_with_embeddings['Hospital'].isnull() == False]
    df_temp = df_temp.loc[df_temp['Hospital'] == k]
    auc = roc_auc_score(df_temp['Positive'], df_temp['probs'])
    print(k, auc)


ERROR! Session/line number was not unique in database. History logging moved to new session 30
HH 0.6964977983621103
BH 0.6017756272444563
HOCC 0.6317667745288713
MMC 0.6128701168679171


In [21]:
for k in ["HH", "BH", "HOCC", "MMC"]:
    df = pd.read_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_{k}_raw_notes.csv')
    df = pd.merge(df, df_with_embeddings[['PatientDurableKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'] + list(notes_embeddings.columns)])
    df.to_csv(f'/datafs_a/carolgao/haim14_data/blood_infection/base_{k}_emb_new.csv', index = False)

In [181]:
df_with_embeddings = pd.concat([df_notes, notes_embeddings], axis=1)
base_train = pd.merge(train_all, df_with_embeddings.drop(columns = {'Positive_bin', 'CombinedNotes'}), on = ['PatientKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], how ='left')
base_test = pd.merge(test_all, df_with_embeddings.drop(columns = {'Positive_bin', 'CombinedNotes'}), on = ['PatientKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], how ='left')

In [184]:
assert len(base_train) == len(train_all)
assert len(base_test) == len(test_all)

In [185]:
base_train = base_train.drop(columns = {"Unnamed: 0.2", "Unnamed: 0.1", "Unnamed: 0"})
base_test = base_test.drop(columns = {"Unnamed: 0.2", "Unnamed: 0.1", "Unnamed: 0"})

In [187]:
base_train.to_csv('/datafs_a/haim14_data/blood_infection/base_train.csv', index = False)
base_test.to_csv('/datafs_a/haim14_data/blood_infection/base_test.csv', index = False)

In [12]:
predicted_probs = pd.read_csv('/datafs_a/haim14_data/blood_infection/notes_predicted_probs.csv')
predicted_probs= predicted_probs.rename(columns = {"0": "pred_probs"})

In [17]:
base_train = pd.read_csv('/datafs_a/haim14_data/blood_infection/base_train.csv')
base_test = pd.read_csv('/datafs_a/haim14_data/blood_infection/base_test.csv')

In [19]:
df_pred = pd.concat([df_notes, predicted_probs], axis = 1)
base_train = pd.merge(base_train, df_pred.drop(columns = {'Positive_bin', 'CombinedNotes'}), on = ['PatientKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], how ='left')
base_test = pd.merge(base_test, df_pred.drop(columns = {'Positive_bin', 'CombinedNotes'}), on = ['PatientKey', 'EncounterKey', 'OrderedDateKey', 'OrderedTimeOfDay'], how ='left')

In [20]:
base_train.to_csv('/datafs_a/haim14_data/blood_infection/base_train.csv', index = False)
base_test.to_csv('/datafs_a/haim14_data/blood_infection/base_test.csv', index = False)

In [22]:
roc_auc_score(test_pred['Positive_bin'], test_pred['pred_probs'])

0.7463929734227188

In [2]:
base_train = pd.read_csv('/datafs_a/haim14_data/blood_infection/base_train.csv')
base_test = pd.read_csv('/datafs_a/haim14_data/blood_infection/base_test.csv')

In [7]:
base_train

,PatientKey,EncounterKey,OrderedDateKey,Positive_bin,BothPositive,AnyOrganism,FalsePositive,OrderedTimeOfDay,SpO2,Resp,...,emb_759,emb_760,emb_761,emb_762,emb_763,emb_764,emb_765,emb_766,emb_767,pred_probs
0,100044774,161330092,20210413,0,0,0,False,2154,97.0,18.0,...,0.552495,-0.302689,0.139551,-0.162922,0.052355,0.136776,-0.036263,-0.106109,0.399403,0.020233
1,100286331,198470410,20220208,0,0,0,False,144,97.0,18.0,...,0.574985,-0.294348,0.136146,-0.205350,0.056274,0.097459,-0.027851,-0.136274,0.396378,0.018162
2,100286331,177232693,20210809,0,0,0,False,36,99.0,20.0,...,0.433561,-0.185002,0.113411,-0.236715,0.149983,0.031856,-0.218985,-0.053494,0.222941,0.039605
3,100286331,198470410,20220126,0,0,0,False,1503,97.0,18.0,...,0.576168,-0.302806,0.134427,-0.162199,0.042622,0.128756,0.009167,-0.139192,0.421601,0.113593
4,100286331,198470410,20220129,0,0,0,False,554,97.0,18.0,...,0.576168,-0.302806,0.134427,-0.162199,0.042622,0.128756,0.009167,-0.139192,0.421601,0.116763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69781,52128119,122959588,20200622,0,0,0,False,1001,100.0,16.0,...,0.545216,-0.290464,0.112192,-0.138026,0.061905,0.143211,-0.007518,-0.125805,0.397461,0.015359
69782,52128119,122959588,20200621,0,0,0,False,1210,100.0,16.0,...,0.545216,-0.290464,0.112192,-0.138026,0.061905,0.143211,-0.007518,-0.125805,0.397461,0.016163
69783,52128119,101537437,20191102,0,0,0,False,300,100.0,20.0,...,0.549146,-0.311043,0.113822,-0.143789,0.060746,0.132086,0.000771,-0.134836,0.409778,0.089354
69784,52128119,122959588,20200624,0,0,0,False,1246,100.0,16.0,...,0.545216,-0.290464,0.112192,-0.138026,0.061905,0.143211,-0.007518,-0.125805,0.397461,0.115579


In [6]:
base_train['Positive_bin'].mean()

0.04263032699968475

In [10]:
base_train = base_train.sort_values(by = ['EncounterKey', 'OrderedDateKey'])
base_train.groupby('EncounterKey').first()['Positive_bin'].mean()

0.04095512125854789